# Módulo 23 · Práctica: abrir la caja negra

Ves en miniatura, con datos cuya verdad conoces, tres herramientas de la interpretabilidad: **superposición**, **autoencoders dispersos** y **sondas lineales**. Así puedes comprobar si cada una encuentra lo que hay.

**Cómo trabajar:** completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás. Todo corre en CPU en menos de 3 minutos.

| Parte | Qué haces |
| --- | --- |
| 1 | Flujo residual y *logit lens* con el ejemplo del documento |
| 2 | Modelo de juguete de superposición: 5 características en 2 dimensiones |
| 3 | Autoencoder disperso que recupera 16 características conocidas |
| 4 | Sonda lineal y *steering* con la dirección encontrada |
| 5 | Calibración: el error de calibración esperado (ECE) |
| 6 | Retos |

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.set_num_threads(1)     # los modelos son diminutos: un hilo basta
torch.set_printoptions(precision=4, sci_mode=False)
print("PyTorch", torch.__version__)

## Parte 1 · Flujo residual y logit lens

Cada capa **suma** su salida al flujo residual: $x_{\text{final}} = x_0 + a_1 + m_1$. Como la desincrustación $W_U$ es lineal, los logits finales son la suma de los logits que aporta cada pieza. El *logit lens* aplica $W_U$ (y la softmax) al flujo **parcial** tras cada capa para ver cómo evoluciona la predicción.

Usamos el ejemplo del documento (sección 23.1): $d = 2$, vocabulario de 3 tokens.

In [ ]:
WU = torch.tensor([[1., 0.], [0., 1.], [-1., 1.]])   # 3 tokens × 2 dimensiones
x0 = torch.tensor([1., 0.])      # embedding
a1 = torch.tensor([0.5, 1.])     # salida de la atención
m1 = torch.tensor([-0.5, 0.5])   # salida del MLP
print("logits finales:", WU @ (x0 + a1 + m1))
print("suma de aportaciones:", WU @ x0 + WU @ a1 + WU @ m1)

**Ejercicio 1.** Completa `logit_lens(aportaciones, WU)`: recibe la lista `[x0, a1, m1, ...]` y devuelve un tensor con una fila por etapa; la fila k es la softmax de $W_U$ aplicado a la suma de las k + 1 primeras aportaciones.

In [ ]:
def logit_lens(aportaciones, WU):
    filas = []
    flujo = torch.zeros_like(aportaciones[0])
    for a in aportaciones:
        # TU CÓDIGO: acumula a en el flujo y guarda softmax(WU @ flujo)
        pass
    return torch.stack(filas)

In [ ]:
# Comprobación: valores de la tabla de la sección 23.1
P = logit_lens([x0, a1, m1], WU)
print(P)
assert P.shape == (3, 3), "Una fila por etapa y una columna por token"
assert torch.allclose(P[:, 0], torch.tensor([0.6652, 0.5741, 0.3072]), atol=1e-3), "Revisa la acumulación del flujo"
assert P[-1].argmax() == 1, "Tras el MLP el token más probable debe ser el 1"
print("Correcto: la predicción cambia del token 0 al token 1 en la última capa.")

## Parte 2 · Superposición: 5 características en 2 dimensiones

Reproducimos el modelo de juguete de *Toy Models of Superposition* (Elhage et al., Anthropic, 2022). Hay $n = 5$ características y solo $m = 2$ neuronas ocultas:

$$h = W x, \qquad \hat x = \mathrm{ReLU}(W^\top h + b) = \mathrm{ReLU}(W^\top W x + b)$$

Cada característica vale 0 con probabilidad $S$ (la **dispersión**) y, si no, un número uniforme en $[0, 1]$. La pérdida pondera cada característica por su importancia $I_i = 0{,}9^i$:

$$L = \mathbb{E}\Big[\sum_i I_i\,(x_i - \hat x_i)^2\Big]$$

La columna $i$ de $W$ es el vector con el que se guarda la característica $i$ en el plano. Entrenamos tres modelos a la vez, con $S = 0$, $0{,}7$ y $0{,}95$.

In [ ]:
n, m = 5, 2
imp = 0.9 ** torch.arange(n).float()          # importancias 1; 0,9; 0,81; ...
S = torch.tensor([0.0, 0.7, 0.95])             # dispersión de cada modelo
K = len(S)

def datos(B, S, gen=None):
    """K lotes de B vectores de n características dispersas."""
    valor = torch.rand(len(S), B, n, generator=gen)
    vivo = (torch.rand(len(S), B, n, generator=gen) > S[:, None, None]).float()
    return valor * vivo
print(datos(4, S)[2])   # con S = 0,95 casi todo son ceros

**Ejercicio 2.** Completa `juguete(x, W, b)` (con `W` de forma `(K, m, n)`, `b` de forma `(K, n)` y `x` de forma `(K, B, n)`) y `perdida(x, xh)`, que devuelve un tensor de K pérdidas: la media sobre el lote de $\sum_i I_i (x_i - \hat x_i)^2$.

In [ ]:
def juguete(x, W, b):
    # TU CÓDIGO: h = x Wᵀ (forma K, B, m); x̂ = ReLU(h W + b)
    pass

def perdida(x, xh):
    # TU CÓDIGO: suma ponderada por imp sobre las características, media sobre el lote
    pass

In [ ]:
# Comprobación con el pentágono del documento (sección 23.3)
ang = 2 * math.pi * torch.arange(5) / 5
Wp = torch.stack([torch.cos(ang), torch.sin(ang)])[None]     # (1, 2, 5)
bp = torch.full((1, 5), -0.31)
xp = torch.tensor([[[1., 1., 0., 0., 0.]]])
xh = juguete(xp, Wp, bp)
assert xh.shape == (1, 1, 5), f"Forma esperada (1, 1, 5), sale {tuple(xh.shape)}"
assert torch.allclose(xh, torch.tensor([[[0.999, 0.999, 0., 0., 0.]]]), atol=1e-3), "Revisa h = Wx y la ReLU"
assert abs(perdida(xp, xh).item() - 1.9e-6) < 1e-6, "Revisa la ponderación por imp"
print("Correcto: el pentágono recupera (1, 1, 0, 0, 0) casi sin error.")

Ahora entrenamos los tres modelos con Adam (unos segundos).

In [ ]:
torch.manual_seed(0)
gen = torch.Generator().manual_seed(0)
W = nn.Parameter(0.5 * torch.randn(K, m, n))
b = nn.Parameter(torch.zeros(K, n))
opt = torch.optim.Adam([W, b], lr=1e-2)
for paso in range(2000):
    x = datos(512, S, gen)
    L = perdida(x, juguete(x, W, b))
    opt.zero_grad(); L.sum().backward(); opt.step()
W_ = W.detach(); b_ = b.detach()
print("pérdidas finales:", L.detach())

In [ ]:
fig, ejes = plt.subplots(1, K, figsize=(12, 4))
colores = plt.cm.viridis(torch.linspace(0, 0.9, n).numpy())
for k, ax in enumerate(ejes):
    for i in range(n):
        ax.arrow(0, 0, W_[k, 0, i], W_[k, 1, i], color=colores[i], width=0.02, length_includes_head=True)
        ax.text(1.12 * W_[k, 0, i], 1.12 * W_[k, 1, i], str(i), color=colores[i])
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal")
    ax.set_title(f"S = {S[k]:.2f}"); ax.axhline(0, lw=0.3, c="k"); ax.axvline(0, lw=0.3, c="k")
plt.suptitle("Columnas de W: cada flecha es una característica")
plt.show()

**Ejercicio 3.** Completa `representadas(Wk, umbral)`, que cuenta cuántas columnas de una matriz `Wk` (forma `m × n`) tienen norma mayor que `umbral`, y `interferencia(Wk)`, que devuelve la matriz $W^\top W$ ($n \times n$). Fuera de la diagonal, $W^\top W$ mide cuánto se «pisan» dos características.

In [ ]:
def representadas(Wk, umbral=0.5):
    # TU CÓDIGO
    pass

def interferencia(Wk):
    # TU CÓDIGO
    pass

In [ ]:
# Comprobación
cuentas = [representadas(W_[k]) for k in range(K)]
print("características representadas por modelo:", cuentas)
assert interferencia(W_[0]).shape == (n, n), "WᵀW debe ser n × n"
assert cuentas[0] == 2, "Sin dispersión el modelo guarda solo las 2 más importantes"
assert cuentas[2] == 5, "Con S = 0,95 el modelo debería meter las 5 en superposición"
assert cuentas[0] <= cuentas[1] <= cuentas[2], "Más dispersión, más características"
G = interferencia(W_[2])
print("WᵀW con S = 0,95 (fuera de la diagonal, interferencias):\n", G)
print("sesgos con S = 0,95:", b_[2])
print("Correcto: la dispersión permite la superposición; los sesgos negativos filtran la interferencia.")

**Qué observar.** Con $S = 0$ el modelo hace PCA: guarda las dos más importantes, ortogonales. Con $S = 0{,}95$ coloca las cinco en un **pentágono** y usa sesgos negativos para que la ReLU borre la interferencia. Con $S = 0{,}7$ aparece un estado intermedio: guarda 4 de las 5 como dos **pares antipodales** ($w_i ≈ -w_j$) en direcciones perpendiculares; en esta semilla queda fuera la característica 2.

## Parte 3 · Autoencoder disperso (SAE)

Fabricamos «activaciones» en $d = 8$ dimensiones como combinaciones dispersas de **16 características conocidas** (vectores unitarios aleatorios: superposición). Cada una se activa con probabilidad 0,06 y amplitud uniforme en $[0, 2]$.

El SAE tiene $F = 32$ latentes:

$$f = \mathrm{ReLU}\big(W_e^\top (x - b_d) + b_e\big), \qquad \hat x = W_d^\top f + b_d$$

$$L = \lVert x - \hat x \rVert^2 + \lambda \sum_j f_j \,\lVert d_j \rVert$$

donde $d_j$ es la fila $j$ de $W_d$. El factor $\lVert d_j\rVert$ impide la trampa de encoger $f$ y agrandar $d_j$.

In [ ]:
d, F_true, N = 8, 16, 20000
g = torch.Generator().manual_seed(0)
D_true = torch.randn(F_true, d, generator=g)
D_true = D_true / D_true.norm(dim=1, keepdim=True)          # 16 direcciones unitarias
act = (torch.rand(N, F_true, generator=g) < 0.06).float() * 2 * torch.rand(N, F_true, generator=g)
X = act @ D_true + 0.01 * torch.randn(N, d, generator=g)
print("características activas por ejemplo (media):", (act > 0).sum(1).float().mean().item())
C = D_true @ D_true.T - torch.eye(F_true)
print("máximo |coseno| entre características verdaderas:", C.abs().max().item())

**Ejercicio 4.** Completa `forward` (devuelve `f` y `x̂`) y `perdida` (media sobre el lote de la reconstrucción más la penalización L1 ponderada).

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, F):
        super().__init__()
        Wd = torch.randn(F, d); Wd = Wd / Wd.norm(dim=1, keepdim=True)
        self.Wd = nn.Parameter(Wd.clone())        # F × d, filas = direcciones
        self.We = nn.Parameter(Wd.T.clone())      # d × F, empieza como la traspuesta
        self.be = nn.Parameter(torch.zeros(F))
        self.bd = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        # TU CÓDIGO: f = ReLU((x - bd) We + be); x̂ = f Wd + bd
        pass

    def perdida(self, x, lam):
        f, xh = self(x)
        # TU CÓDIGO: ||x - x̂||² por ejemplo + lam · Σ_j f_j ||d_j||, media sobre el lote
        pass

In [ ]:
# Comprobación rápida de formas y de la pérdida en un caso controlado
torch.manual_seed(0)
sae_p = SAE(d, 32)
f, xh = sae_p(X[:5])
assert f.shape == (5, 32) and xh.shape == (5, d), "Formas: f (B, F) y x̂ (B, d)"
assert (f >= 0).all(), "f debe pasar por una ReLU"
with torch.no_grad():
    sae_p.We.zero_(); sae_p.be.zero_(); sae_p.bd.zero_()
assert abs(sae_p.perdida(X[:5], 0.3).item() - (X[:5] ** 2).sum(1).mean().item()) < 1e-5, \
    "Con f = 0 la pérdida es la media de ||x||²"
print("Correcto.")

In [ ]:
def entrena_sae(lam, pasos=2500, semilla=0):
    torch.manual_seed(semilla)
    sae = SAE(d, 32)
    opt = torch.optim.Adam(sae.parameters(), lr=3e-3)
    for _ in range(pasos):
        x = X[torch.randint(0, N, (256,))]
        L = sae.perdida(x, lam)
        opt.zero_grad(); L.backward(); opt.step()
    return sae

sae = entrena_sae(lam=0.3)
with torch.no_grad():
    f, Xh = sae(X)
fvu = (((X - Xh) ** 2).sum() / ((X - X.mean(0)) ** 2).sum()).item()
L0 = (f > 0).float().sum(1).mean().item()
print(f"varianza no explicada: {fvu:.3f} · latentes activos por ejemplo (L0): {L0:.2f}")

**Ejercicio 5.** Completa `recuperacion(D_true, Wd)`: normaliza las filas de `Wd`, calcula la matriz de cosenos (16 × 32) y devuelve, para cada característica verdadera, el **máximo** coseno con algún latente del SAE.

In [ ]:
def recuperacion(D_true, Wd):
    # TU CÓDIGO
    pass

In [ ]:
# Comprobación
mejor = recuperacion(D_true, sae.Wd.detach())
print("mejor coseno por característica:", mejor)
assert mejor.shape == (F_true,), "Un valor por característica verdadera"
assert (mejor > 0.9).sum() >= 14, f"El SAE debería recuperar al menos 14 de 16; recupera {(mejor > 0.9).sum().item()}"
assert fvu < 0.15 and L0 < 4, "Reconstrucción razonable y código disperso"
print(f"Correcto: {(mejor > 0.9).sum().item()} de 16 características recuperadas con coseno > 0,9.")

**El compromiso de λ.** Repite con una penalización pequeña y una grande. Con λ pequeño la reconstrucción es casi perfecta, pero se encienden unos 9 latentes cuando de media hay 1 característica activa. Con λ grande el código es muy disperso y la reconstrucción empeora (FVU ≈ 0,2). En SAE reales, sin verdad conocida, λ se elige con esta curva L0–FVU.

In [ ]:
for lam in [0.03, 1.0]:
    s = entrena_sae(lam, pasos=1500)
    with torch.no_grad():
        f_, Xh_ = s(X)
    fv = (((X - Xh_) ** 2).sum() / ((X - X.mean(0)) ** 2).sum()).item()
    rec = (recuperacion(D_true, s.Wd.detach()) > 0.9).sum().item()
    print(f"λ = {lam:4}: FVU = {fv:.3f} · L0 = {(f_ > 0).float().sum(1).mean():.2f} · recuperadas = {rec}/16")

## Parte 4 · Sonda lineal y steering

Simulamos un flujo residual de $d = 32$ dimensiones. Cada vector mezcla 10 características dispersas de «ruido semántico», ruido gaussiano y un **concepto binario** $y$ (por ejemplo, «la frase está en pasado»), guardado como $\pm 1$ veces una dirección $c$ que tú no conoces de antemano. Una sonda lineal es una regresión logística $p(y = 1 \mid x) = \sigma(w^\top x + w_0)$.

In [ ]:
dr, Np = 32, 4000
g = torch.Generator().manual_seed(1)
c_true = torch.randn(dr, generator=g); c_true = c_true / c_true.norm()
U = torch.randn(10, dr, generator=g); U = U / U.norm(dim=1, keepdim=True)
y = (torch.rand(Np, generator=g) < 0.5).float()
otras = (torch.rand(Np, 10, generator=g) < 0.3).float() * torch.rand(Np, 10, generator=g) * 2
R = (2 * y - 1)[:, None] * c_true + otras @ U + 0.5 * torch.randn(Np, dr, generator=g)
Rtr, ytr, Rte, yte = R[:3000], y[:3000], R[3000:], y[3000:]
print(R.shape, "fracción de positivos:", y.mean().item())

**Ejercicio 6.** Entrena la sonda con `nn.Linear(dr, 1)`, `BCEWithLogitsLoss` y Adam (lr = 0,05, 300 pasos con todo el conjunto de entrenamiento). Guarda en `acc` la exactitud sobre el conjunto de prueba.

In [ ]:
torch.manual_seed(0)
sonda = nn.Linear(dr, 1)
# TU CÓDIGO: bucle de entrenamiento y exactitud en prueba
acc = None

In [ ]:
# Comprobación
w = sonda.weight.detach().squeeze(0)
cos_sonda = (w @ c_true / w.norm()).item()
mu = Rtr[ytr == 1].mean(0) - Rtr[ytr == 0].mean(0)
cos_medias = (mu @ c_true / mu.norm()).item()
print(f"exactitud en prueba: {acc:.3f}")
print(f"coseno(sonda, c) = {cos_sonda:.3f} · coseno(diferencia de medias, c) = {cos_medias:.3f}")
assert acc is not None and acc > 0.95, "La sonda debería superar el 95 % de exactitud"
assert cos_sonda > 0.8, "La dirección de la sonda debería parecerse a la verdadera"
print("Correcto: el concepto es linealmente decodificable.")

**Control.** La misma sonda con etiquetas **aleatorias** debe quedarse cerca del 50 % en prueba; si no, mide su propia capacidad, no lo que el modelo representa.

In [ ]:
torch.manual_seed(0)
y_azar = (torch.rand(Np) < 0.5).float()
ctrl = nn.Linear(dr, 1); o = torch.optim.Adam(ctrl.parameters(), lr=0.05)
for _ in range(300):
    L = nn.functional.binary_cross_entropy_with_logits(ctrl(Rtr).squeeze(1), y_azar[:3000])
    o.zero_grad(); L.backward(); o.step()
with torch.no_grad():
    acc_ctrl = ((ctrl(Rte).squeeze(1) > 0).float() == y_azar[3000:]).float().mean().item()
print(f"exactitud con etiquetas aleatorias: {acc_ctrl:.3f}")

**Ejercicio 7 (steering).** Toma los ejemplos de prueba con $y = 0$ y súmales $\alpha\,\hat\mu$, donde $\hat\mu$ es la diferencia de medias normalizada. Completa `fraccion_volteada(alpha)`: la fracción de esos ejemplos que la sonda clasifica ahora como $y = 1$.

In [ ]:
mu_hat = mu / mu.norm()
negativos = Rte[yte == 0]

def fraccion_volteada(alpha):
    # TU CÓDIGO
    pass

In [ ]:
# Comprobación
alfas = [0, 0.5, 1, 1.5, 2, 3]
fr = [fraccion_volteada(a) for a in alfas]
print([f"{f:.2f}" for f in fr])
assert fr[0] < 0.05, "Sin steering casi ninguno debe cambiar"
assert fr[-1] > 0.95, "Con α = 3 casi todos deberían cambiar"
assert all(a <= b + 1e-9 for a, b in zip(fr, fr[1:])), "La fracción debe crecer con α"
plt.plot(alfas, fr, "o-"); plt.xlabel("α"); plt.ylabel("fracción clasificada y = 1")
plt.title("Steering con la dirección del concepto"); plt.show()
print("Correcto: sumar la dirección cambia lo que el resto del modelo «lee».")

## Parte 5 · Calibración

Un modelo está **calibrado** si, entre las respuestas a las que asigna confianza 0,8, acierta el 80 %. El error de calibración esperado agrupa las predicciones en cubos de confianza:

$$\mathrm{ECE} = \sum_b \frac{n_b}{n}\,\big|\,\mathrm{conf}_b - \mathrm{acc}_b\,\big|$$

**Ejercicio 8.** Completa `ece(conf, acierto, cubos)`: `conf` es un tensor de confianzas en $[0, 1]$, `acierto` un tensor de 0/1 y `cubos` el número de intervalos iguales en $[0, 1]$. Ignora los cubos vacíos.

In [ ]:
def ece(conf, acierto, cubos=10):
    # TU CÓDIGO: para cada cubo [k/cubos, (k+1)/cubos), suma n_b/n · |media conf − media acierto|
    pass

In [ ]:
# Comprobación 1: el ejemplo del documento (sección 23.7): ECE = 0,12
conf = torch.tensor([0.9] * 50 + [0.6] * 30 + [0.3] * 20)
acierto = torch.tensor([1] * 35 + [0] * 15 + [1] * 18 + [0] * 12 + [1] * 8 + [0] * 12)
assert abs(ece(conf, acierto) - 0.12) < 1e-6, f"Esperaba 0,12 y sale {ece(conf, acierto)}"
# Comprobación 2: un modelo calibrado por construcción tiene ECE ≈ 0; uno sobreconfiado, no
g = torch.Generator().manual_seed(0)
p = torch.rand(20000, generator=g)
ok = (torch.rand(20000, generator=g) < p).float()
sobre = torch.clamp(0.5 + 2 * (p - 0.5), 0, 1)    # exagera la confianza
print(f"ECE calibrado: {ece(p, ok):.3f} · ECE sobreconfiado: {ece(sobre, ok):.3f}")
assert ece(p, ok) < 0.02 and ece(sobre, ok) > 0.1
print("Correcto.")

## Parte 6 · Retos

1. **Transición de fase.** Parte 2 con $n = 2$, $m = 1$ e importancias iguales: comprueba que el modelo pasa a guardar las dos en antipodal ($w_1 = -w_2$) cerca de $S = 3/7$ (ejercicio 13 del documento).
2. **Latentes muertos.** SAE con $F = 64$ y λ = 1: cuenta los latentes que nunca se activan.
3. **Sonda frente a diferencia de medias.** Correlaciona el ruido con $c$: ¿qué dirección se aleja más de $c$? ¿Cuál sirve mejor para steering?
4. **Un mini-GPT real.** Guarda el flujo residual del mini-GPT del módulo 15 y entrena, capa a capa, una sonda para «el siguiente carácter es vocal».

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def logit_lens(aportaciones, WU):
#     filas = []
#     flujo = torch.zeros_like(aportaciones[0])
#     for a in aportaciones:
#         flujo = flujo + a
#         filas.append(torch.softmax(WU @ flujo, dim=0))
#     return torch.stack(filas)

# Ejercicio 2
# def juguete(x, W, b):
#     h = x @ W.transpose(1, 2)
#     return torch.relu(h @ W + b[:, None, :])
#
# def perdida(x, xh):
#     return (imp * (x - xh) ** 2).sum(-1).mean(-1)

# Ejercicio 3
# def representadas(Wk, umbral=0.5):
#     return int((Wk.norm(dim=0) > umbral).sum())
#
# def interferencia(Wk):
#     return Wk.T @ Wk

# Ejercicio 4
# class SAE(nn.Module):
#     def __init__(self, d, F):
#         super().__init__()
#         Wd = torch.randn(F, d); Wd = Wd / Wd.norm(dim=1, keepdim=True)
#         self.Wd = nn.Parameter(Wd.clone())
#         self.We = nn.Parameter(Wd.T.clone())
#         self.be = nn.Parameter(torch.zeros(F))
#         self.bd = nn.Parameter(torch.zeros(d))
#
#     def forward(self, x):
#         f = torch.relu((x - self.bd) @ self.We + self.be)
#         return f, f @ self.Wd + self.bd
#
#     def perdida(self, x, lam):
#         f, xh = self(x)
#         rec = ((x - xh) ** 2).sum(1)
#         l1 = (f * self.Wd.norm(dim=1)).sum(1)
#         return (rec + lam * l1).mean()

# Ejercicio 5
# def recuperacion(D_true, Wd):
#     Dn = Wd / Wd.norm(dim=1, keepdim=True)
#     return (D_true @ Dn.T).max(dim=1).values

# Ejercicio 6
# torch.manual_seed(0)
# sonda = nn.Linear(dr, 1)
# opt = torch.optim.Adam(sonda.parameters(), lr=0.05)
# lossf = nn.BCEWithLogitsLoss()
# for _ in range(300):
#     L = lossf(sonda(Rtr).squeeze(1), ytr)
#     opt.zero_grad(); L.backward(); opt.step()
# with torch.no_grad():
#     acc = ((sonda(Rte).squeeze(1) > 0).float() == yte).float().mean().item()

# Ejercicio 7
# mu_hat = mu / mu.norm()
# negativos = Rte[yte == 0]
#
# def fraccion_volteada(alpha):
#     with torch.no_grad():
#         return (sonda(negativos + alpha * mu_hat).squeeze(1) > 0).float().mean().item()

# Ejercicio 8
# def ece(conf, acierto, cubos=10):
#     idx = torch.clamp((conf * cubos).long(), max=cubos - 1)
#     total = 0.0
#     for k in range(cubos):
#         sel = idx == k
#         if sel.any():
#             total += sel.float().mean() * (conf[sel].mean() - acierto[sel].float().mean()).abs()
#     return float(total)
